In [1]:
!pip -q install PyPDF2 python-docx reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.6 MB/s eta 0:00:00


In [2]:
import os

PROJECT_DIR = "/content/LabADV_Week11"

folders = [
    "src",
    "configs",
    "documents",
    "templates",
    "output"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("Project directory:", PROJECT_DIR)
for folder in folders:
    print("Created:", folder)

Project directory: /content/LabADV_Week11
Created: src
Created: configs
Created: documents
Created: templates
Created: output


In [3]:
from reportlab.pdfgen import canvas

DOCS_DIR = os.path.join(PROJECT_DIR, "documents")

def create_sample_pdf(path, title, lines):
    pdf = canvas.Canvas(path)
    pdf.setTitle(title)

    pdf.setFont("Helvetica-Bold", 18)
    pdf.drawString(72, 760, title)

    pdf.setFont("Helvetica", 12)
    y = 720

    for line in lines:
        pdf.drawString(72, y, line)
        y -= 25

    pdf.save()

create_sample_pdf(
    os.path.join(DOCS_DIR, "sample_1.pdf"),
    "Sample Document One",
    [
        "Lab Advanced Week 11",
        "PDF Merge Demonstration",
        "This is the first sample document."
    ]
)

create_sample_pdf(
    os.path.join(DOCS_DIR, "sample_2.pdf"),
    "Sample Document Two",
    [
        "Document Automation",
        "This is the second sample document.",
        "Both documents will be merged into one PDF."
    ]
)

create_sample_pdf(
    os.path.join(DOCS_DIR, "input_article.pdf"),
    "Document Automation Report Source",
    [
        "Document automation reduces repetitive manual work.",
        "Python can extract text and combine PDF documents.",
        "Word reports can present results in a structured format."
    ]
)

print("สร้าง PDF ตัวอย่างครบ 3 ไฟล์แล้ว")
print(os.listdir(DOCS_DIR))

สร้าง PDF ตัวอย่างครบ 3 ไฟล์แล้ว
['input_article.pdf', 'sample_1.pdf', 'sample_2.pdf']


In [4]:
%%writefile /content/LabADV_Week11/src/utils.py

import logging
import os

def setup_logging(name="LabADV_Week11"):
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True
    )
    return logging.getLogger(name)

def ensure_directory_exists(path):
    os.makedirs(path, exist_ok=True)
    return path

Writing /content/LabADV_Week11/src/utils.py


In [5]:
%%writefile /content/LabADV_Week11/src/pdf_processor.py

import io
import os
from PyPDF2 import PdfReader, PdfWriter
from reportlab.pdfgen import canvas
from reportlab.lib.colors import Color


class PDFProcessor:

    def extract_text(self, input_pdf):
        if not os.path.exists(input_pdf):
            raise FileNotFoundError(f"ไม่พบไฟล์: {input_pdf}")

        reader = PdfReader(input_pdf)

        if reader.is_encrypted and not reader.decrypt(""):
            raise ValueError(f"ไม่สามารถเปิด PDF ที่เข้ารหัส: {input_pdf}")

        results = []
        for page_number, page in enumerate(reader.pages, start=1):
            results.append({
                "page": page_number,
                "text": page.extract_text() or ""
            })

        return results

    def merge_pdfs(self, input_files, output_pdf):
        if not input_files:
            raise ValueError("ไม่มีไฟล์ PDF สำหรับรวม")

        writer = PdfWriter()

        for file_path in input_files:
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"ไม่พบไฟล์: {file_path}")

            reader = PdfReader(file_path)

            if reader.is_encrypted and not reader.decrypt(""):
                raise ValueError(f"PDF ถูกเข้ารหัส: {file_path}")

            for page in reader.pages:
                writer.add_page(page)

        os.makedirs(os.path.dirname(output_pdf), exist_ok=True)

        with open(output_pdf, "wb") as file:
            writer.write(file)

        return output_pdf

    def add_watermark(self, input_pdf, output_pdf, watermark_text="CONFIDENTIAL"):
        if not os.path.exists(input_pdf):
            raise FileNotFoundError(f"ไม่พบไฟล์: {input_pdf}")

        reader = PdfReader(input_pdf)
        writer = PdfWriter()

        if reader.is_encrypted and not reader.decrypt(""):
            raise ValueError("ไม่สามารถเปิด PDF ที่เข้ารหัสได้")

        for original_page in reader.pages:
            width = float(original_page.mediabox.width)
            height = float(original_page.mediabox.height)

            packet = io.BytesIO()
            watermark_canvas = canvas.Canvas(
                packet,
                pagesize=(width, height)
            )

            watermark_canvas.saveState()
            watermark_canvas.setFillColor(
                Color(0.65, 0.65, 0.65, alpha=0.45)
            )
            watermark_canvas.setFont("Helvetica-Bold", 30)
            watermark_canvas.translate(width / 2, height / 2)
            watermark_canvas.rotate(35)
            watermark_canvas.drawCentredString(
                0, 0, watermark_text
            )
            watermark_canvas.restoreState()
            watermark_canvas.save()

            packet.seek(0)
            watermark_page = PdfReader(packet).pages[0]

            original_page.merge_page(watermark_page)
            writer.add_page(original_page)

        os.makedirs(os.path.dirname(output_pdf), exist_ok=True)

        with open(output_pdf, "wb") as file:
            writer.write(file)

        return output_pdf

Writing /content/LabADV_Week11/src/pdf_processor.py


In [6]:
%%writefile /content/LabADV_Week11/src/word_processor.py

import os
from docx import Document
from docx.shared import Pt


class WordProcessor:

    def create_report(self, output_path, extracted_pages, merged_pdf_path):
        document = Document()

        document.add_heading(
            "Document Automation Report",
            level=0
        )

        intro = document.add_paragraph()
        run = intro.add_run(
            "Lab Advanced Week 11 - Processing Results"
        )
        run.bold = True
        run.font.size = Pt(12)

        document.add_paragraph(
            "This report summarizes the automated processing "
            "of PDF documents using Python."
        )

        document.add_heading("Extracted PDF Text", level=1)

        for item in extracted_pages:
            document.add_heading(
                f"Page {item['page']}",
                level=2
            )
            document.add_paragraph(
                item["text"] or "(No selectable text found.)"
            )

        document.add_heading("Processing Summary", level=1)

        table = document.add_table(rows=1, cols=2)
        table.style = "Table Grid"

        table.rows[0].cells[0].text = "Task"
        table.rows[0].cells[1].text = "Result"

        results = [
            ("PDF Text Extraction", "Completed"),
            ("PDF Merging", os.path.basename(merged_pdf_path)),
            ("Word Report Generation", "Completed"),
            ("PDF Watermarking", "Completed")
        ]

        for task, result in results:
            cells = table.add_row().cells
            cells[0].text = task
            cells[1].text = result

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        document.save(output_path)

        return output_path

Writing /content/LabADV_Week11/src/word_processor.py


In [7]:
%%writefile /content/LabADV_Week11/src/config_parser.py

import json
import os


class ConfigParser:

    def __init__(self, config_path):
        self.config_path = config_path

    def load_config(self):
        if not os.path.exists(self.config_path):
            raise FileNotFoundError(
                f"ไม่พบไฟล์ตั้งค่า: {self.config_path}"
            )

        with open(self.config_path, "r", encoding="utf-8") as file:
            config = json.load(file)

        required_keys = [
            "input_pdfs",
            "watermark_source",
            "output_files"
        ]

        for key in required_keys:
            if key not in config:
                raise ValueError(
                    f"ไฟล์ตั้งค่าขาดรายการ: {key}"
                )

        return config

Writing /content/LabADV_Week11/src/config_parser.py


In [8]:
%%writefile /content/LabADV_Week11/configs/document_automation_config.json

{
  "input_pdfs": [
    "documents/sample_1.pdf",
    "documents/sample_2.pdf"
  ],
  "watermark_source": "documents/input_article.pdf",
  "watermark_text": "CONFIDENTIAL",
  "output_files": {
    "merged_pdf": "output/merged_docs.pdf",
    "report_docx": "output/generate_report.docx",
    "watermarked_pdf": "output/watermarked_document.pdf"
  }
}

Writing /content/LabADV_Week11/configs/document_automation_config.json


In [9]:
%%writefile /content/LabADV_Week11/main.py

import os
from src.utils import setup_logging, ensure_directory_exists
from src.config_parser import ConfigParser
from src.pdf_processor import PDFProcessor
from src.word_processor import WordProcessor


def main():
    base_dir = os.path.dirname(os.path.abspath(__file__))
    logger = setup_logging("main")

    config_path = os.path.join(
        base_dir,
        "configs",
        "document_automation_config.json"
    )

    config = ConfigParser(config_path).load_config()

    output_dir = os.path.join(base_dir, "output")
    ensure_directory_exists(output_dir)

    pdf_processor = PDFProcessor()
    word_processor = WordProcessor()

    # 1. รวม PDF
    input_pdfs = [
        os.path.join(base_dir, path)
        for path in config["input_pdfs"]
    ]

    merged_pdf = os.path.join(
        base_dir,
        config["output_files"]["merged_pdf"]
    )

    pdf_processor.merge_pdfs(input_pdfs, merged_pdf)
    logger.info("สร้าง merged_docs.pdf สำเร็จ")

    # 2. ดึงข้อความจาก PDF ต้นฉบับ
    watermark_source = os.path.join(
        base_dir,
        config["watermark_source"]
    )

    extracted_pages = pdf_processor.extract_text(watermark_source)

    # 3. สร้างรายงาน Word
    report_docx = os.path.join(
        base_dir,
        config["output_files"]["report_docx"]
    )

    word_processor.create_report(
        report_docx,
        extracted_pages,
        merged_pdf
    )
    logger.info("สร้าง generate_report.docx สำเร็จ")

    # 4. เพิ่มลายน้ำให้ PDF
    watermarked_pdf = os.path.join(
        base_dir,
        config["output_files"]["watermarked_pdf"]
    )

    pdf_processor.add_watermark(
        watermark_source,
        watermarked_pdf,
        config.get("watermark_text", "CONFIDENTIAL")
    )
    logger.info("สร้าง watermarked_document.pdf สำเร็จ")

    print("\n===== LAB ADV WEEK 11: RESULTS =====")
    for path in [merged_pdf, report_docx, watermarked_pdf]:
        print(("OK" if os.path.exists(path) else "MISSING"), path)


if __name__ == "__main__":
    main()

Writing /content/LabADV_Week11/main.py


In [10]:
%cd /content/LabADV_Week11
!python main.py

/content/LabADV_Week11
2026-10-12 16:21:39,564 - INFO - สร้าง merged_docs.pdf สำเร็จ
2026-10-12 16:21:39,615 - INFO - สร้าง generate_report.docx สำเร็จ
2026-10-12 16:21:39,625 - INFO - สร้าง watermarked_document.pdf สำเร็จ

===== LAB ADV WEEK 11: RESULTS =====
OK /content/LabADV_Week11/output/merged_docs.pdf
OK /content/LabADV_Week11/output/generate_report.docx
OK /content/LabADV_Week11/output/watermarked_document.pdf


In [11]:
import os
from PyPDF2 import PdfReader
from docx import Document

OUTPUT_DIR = "/content/LabADV_Week11/output"

required_files = [
    "merged_docs.pdf",
    "generate_report.docx",
    "watermarked_document.pdf"
]

print("===== ตรวจสอบไฟล์ส่งงาน =====")

for filename in required_files:
    path = os.path.join(OUTPUT_DIR, filename)

    if not os.path.exists(path):
        print(f"❌ ไม่พบไฟล์: {filename}")
        continue

    print(f"✅ {filename} ({os.path.getsize(path):,} bytes)")

    if filename.endswith(".pdf"):
        pdf = PdfReader(path)
        print(f"   จำนวนหน้า: {len(pdf.pages)}")

    elif filename.endswith(".docx"):
        doc = Document(path)
        print(f"   จำนวนย่อหน้า: {len(doc.paragraphs)}")
        print(f"   จำนวนตาราง: {len(doc.tables)}")

===== ตรวจสอบไฟล์ส่งงาน =====
✅ merged_docs.pdf (2,087 bytes)
   จำนวนหน้า: 2
✅ generate_report.docx (37,070 bytes)
   จำนวนย่อหน้า: 7
   จำนวนตาราง: 1
✅ watermarked_document.pdf (2,264 bytes)
   จำนวนหน้า: 1


In [12]:
from PyPDF2 import PdfReader

watermarked_path = (
    "/content/LabADV_Week11/output/watermarked_document.pdf"
)

reader = PdfReader(watermarked_path)

print("จำนวนหน้า:", len(reader.pages))
print("ข้อความหน้าแรก:")
print(reader.pages[0].extract_text() or "(ไม่พบข้อความที่ดึงได้)")
print("\nหมายเหตุ: ตรวจลายน้ำด้วยสายตาอีกครั้ง")

จำนวนหน้า: 1
ข้อความหน้าแรก:
Document Automation Report Source
Document automation reduces repetitive manual work.
Python can extract text and combine PDF documents.
Word reports can present results in a structured format.
CONFIDENTIAL


หมายเหตุ: ตรวจลายน้ำด้วยสายตาอีกครั้ง


In [13]:
from google.colab import files
import os

output_dir = "/content/LabADV_Week11/output"

for filename in [
    "merged_docs.pdf",
    "generate_report.docx",
    "watermarked_document.pdf"
]:
    path = os.path.join(output_dir, filename)

    if os.path.exists(path):
        files.download(path)
    else:
        print("ไม่พบไฟล์:", filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>